# Kaggle runner — DL-PRJ04

Clona el repo, monta ASVspoof 2019, entrena y evalúa. Acelerador: **GPU T4** (no P100).

**Datasets (Add data):** `asvpoof-2019-dataset` (público, p.ej. awsaf49). Opcionales: asvspoof2021-la, mlaad-es, voces-propias.

**Lecciones ya aplicadas (no reintroducir):**
- Buscar ASV19 en `/kaggle/input/datasets/...` y en la ruta corta.
- `build_protocols.py --skip-file-check` (el check de cada FLAC cuelga en el FS de Kaggle).
- `num_workers: 0` en train y eval (LibsndfileError con workers).
- LFCC en fp32 (ya en `main`); no entrenar con el LFCC antiguo en fp16.
- Logs con `flush=True` / progreso por batch (el log de Kaggle bufferiza).

In [ ]:
# === Parámetros de la corrida ===
REPO_URL = "https://github.com/diegoanro22/DL-PRJ04.git"
BRANCH = "main"
CONFIG = "configs/lfcc_cnn.yaml"   # A1; A2: lfcc_cnn_aug; B1: aasist_l; B2: aasist_l_aug
RUN_NAME = "A1"
MACHINE = "kaggle"
EPOCHS = None   # None = usa el del config
RESUME = False  # True para retomar desde /kaggle/working/runs/<RUN_NAME>
EVAL_SPLITS = ["asv19_dev", "asv19_eval"]

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "DL-PRJ04"
INP = Path("/kaggle/input")

print("input:", [p.name for p in INP.iterdir()] if INP.exists() else None, flush=True)

def find_asv19() -> Path:
    """Rutas posibles del dataset público (corta o /datasets/owner/slug)."""
    cands = [
        INP / "datasets" / "awsaf49" / "asvpoof-2019-dataset" / "LA" / "LA",
        INP / "asvpoof-2019-dataset" / "LA" / "LA",
    ]
    ds = INP / "datasets"
    if ds.is_dir():
        for owner in ds.iterdir():
            if not owner.is_dir():
                continue
            for slug in owner.iterdir():
                if not slug.is_dir():
                    continue
                name = slug.name.lower()
                if "asvpoof" in name or "asvspoof" in name:
                    cands += [slug / "LA" / "LA", slug / "LA"]
    for p in cands:
        if (p / "ASVspoof2019_LA_cm_protocols").is_dir() and (p / "ASVspoof2019_LA_train" / "flac").is_dir():
            return p
    raise FileNotFoundError(
        "ASVspoof 2019 LA no encontrado. Monta asvpoof-2019-dataset. Candidatos: " + ", ".join(map(str, cands))
    )

ASV19 = find_asv19()
print("ASV19:", ASV19, flush=True)

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(), flush=True)

# Fija la ruta detectada en el machine config (sin rglob profundo).
machine = Path("configs/machines/kaggle.yaml")
lines = []
for line in machine.read_text().splitlines(True):
    if line.strip().startswith("asv19:"):
        lines.append(f"  asv19: {ASV19}\n")
    elif line.strip().startswith("num_workers:"):
        lines.append("num_workers: 0\n")
    else:
        lines.append(line)
machine.write_text("".join(lines))
print("kaggle.yaml asv19 + num_workers=0 OK", flush=True)

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, flush=True)

In [ ]:
import subprocess
print("installing deps...", flush=True)
subprocess.check_call([
    "pip", "install", "-q",
    "pandas", "scikit-learn", "soundfile", "librosa", "pyyaml",
    "tqdm", "tensorboard", "matplotlib",
])
print("deps ok", flush=True)

In [ ]:
import subprocess
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path("configs/machines/kaggle.yaml").read_text())
root = str(Path(cfg["dataset_roots"]["asv19"]))
print("build_protocols --skip-file-check", root, flush=True)
subprocess.check_call([
    "python", "scripts/build_protocols.py", "--root", root, "--skip-file-check",
])
subprocess.check_call(["wc", "-l"] + sorted(map(str, Path("protocols").glob("asv19_*.csv"))))

In [ ]:
import subprocess
from pathlib import Path

if RESUME:
    cmd = ["python", "-m", "src.train", "--resume", f"/kaggle/working/runs/{RUN_NAME}"]
else:
    cmd = ["python", "-m", "src.train", "--config", CONFIG, "--machine", MACHINE, "--name", RUN_NAME]
    if EPOCHS:
        cmd += ["--epochs", str(EPOCHS)]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd)
assert Path(f"/kaggle/working/runs/{RUN_NAME}/best.pt").is_file()

In [ ]:
import subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
ckpt = WORK / "runs" / RUN_NAME / "best.pt"
assert ckpt.is_file(), ckpt

cmd = [
    "python", "-m", "src.evaluate",
    "--checkpoint", str(ckpt),
    "--splits", *EVAL_SPLITS,
    "--machine", MACHINE,
    "--name", RUN_NAME,
]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd)

out = WORK / f"{RUN_NAME}_artifacts"
if out.exists():
    shutil.rmtree(out)
out.mkdir()
shutil.copy2(ckpt, out / "best.pt")
for src in (Path("results/scores"), Path("results/metrics"), Path("results/runs.csv")):
    dest = out / src.name
    if src.is_dir():
        shutil.copytree(src, dest)
    elif src.is_file():
        shutil.copy2(src, dest)
print("Listo", out, flush=True)
subprocess.check_call(["find", str(out), "-type", "f"])